## CBIS-DDSM Breast Cancer Classification — Refined Notebook

Run the cells **top to bottom** after attaching the Kaggle CBIS-DDSM dataset and enabling a GPU.

1. Dataset audit and clean manifests.
2. Patient-level train/validation/test split.
3. Self-contained DenseNet-121 data pipeline and training.

The test split is not used for checkpoint selection. The task is **benign vs malignant among case-linked abnormality images**; benign does not mean normal. High validation performance is not a guarantee of clinical reliability.


In [9]:
# ================================================================
# CBIS-DDSM COMPLETE DATASET PREPARATION
# From original dataset -> image inventory -> role mapping ->
# label conflict audit -> deduplication -> clean manifests.
#
# This cell intentionally stops BEFORE train/validation/test splitting.
# The next step will be patient-level dataset distribution.
# No model training is performed.
# ================================================================

import os
import hashlib
import pandas as pd
from pathlib import Path
from collections import defaultdict

# -------------------- 1. CONFIGURATION --------------------

ROOT = Path(
    "/kaggle/input/datasets/awsaf49/"
    "cbis-ddsm-breast-cancer-image-dataset"
)
JPEG_DIR = ROOT / "jpeg"
CSV_DIR = ROOT / "csv"

OUT = Path("/kaggle/working/cbis_ddsm_preparation")
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("CBIS-DDSM DATASET PREPARATION")
print("=" * 70)

if not JPEG_DIR.exists() or not CSV_DIR.exists():
    raise FileNotFoundError(
        "Dataset not found. Check the Kaggle input path and attach "
        "the awsaf49 CBIS-DDSM dataset to this notebook."
    )

# -------------------- 2. INVENTORY ALL JPEG FILES --------------------

jpeg_files = sorted(
    p for p in JPEG_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg"}
)

if not jpeg_files:
    raise RuntimeError("No JPEG images found in the dataset.")

inventory = pd.DataFrame({
    "image_path": [str(p) for p in jpeg_files],
    "filename": [p.name for p in jpeg_files],
    "series_uid": [p.parent.name for p in jpeg_files],
    "file_size_bytes": [p.stat().st_size for p in jpeg_files],
})

# Map SeriesInstanceUID -> actual JPEG paths
uid_to_jpegs = defaultdict(list)
for p in jpeg_files:
    uid_to_jpegs[p.parent.name].append(str(p))

print(f"\nTotal JPEG files discovered: {len(inventory):,}")

# -------------------- 3. LOAD DICOM METADATA --------------------

dicom_csv = CSV_DIR / "dicom_info.csv"
if not dicom_csv.exists():
    raise FileNotFoundError(f"Missing metadata CSV: {dicom_csv}")

dicom = pd.read_csv(dicom_csv)
dicom.columns = [str(c).strip() for c in dicom.columns]

def find_column(frame, names):
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

uid_col = find_column(
    dicom, ["SeriesInstanceUID", "series_instance_uid"]
)
desc_col = find_column(
    dicom, ["SeriesDescription", "series description"]
)
image_col = find_column(
    dicom, ["image_path", "image path"]
)

if uid_col is None or desc_col is None:
    raise ValueError(
        "Could not find SeriesInstanceUID and SeriesDescription "
        f"in dicom_info.csv. Columns: {dicom.columns.tolist()}"
    )

# Store the metadata description and filename for each series UID.
metadata_by_uid = defaultdict(list)

for _, row in dicom.iterrows():
    uid = str(row[uid_col]).strip() if pd.notna(row[uid_col]) else ""
    desc = (
        str(row[desc_col]).strip().lower()
        if pd.notna(row[desc_col]) else ""
    )
    meta_path = (
        str(row[image_col])
        if image_col and pd.notna(row[image_col]) else ""
    )
    filename = os.path.basename(meta_path.replace("\\", "/"))

    if uid:
        metadata_by_uid[uid].append({
            "description": desc,
            "filename": filename,
            "metadata_path": meta_path,
        })

# Categorize the complete JPEG inventory using metadata descriptions.
def classify_description(description):
    d = str(description).lower()

    matches = []
    if "full mammogram images" in d:
        matches.append("full")
    if "cropped images" in d:
        matches.append("crop")
    if "roi mask images" in d:
        matches.append("roi")

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        return "ambiguous"
    return "unclassified"

uid_group_map = {}
for uid, rows in metadata_by_uid.items():
    groups = {
        classify_description(r["description"])
        for r in rows
    }
    groups.discard("unclassified")

    if len(groups) == 1:
        uid_group_map[uid] = next(iter(groups))
    elif len(groups) > 1:
        uid_group_map[uid] = "ambiguous"
    else:
        uid_group_map[uid] = "unclassified"

inventory["image_group"] = inventory["series_uid"].map(
    uid_group_map
).fillna("unclassified")

inventory.to_csv(
    OUT / "cbis_all_jpeg_inventory.csv", index=False
)

print("\nJPEG inventory by metadata group:")
print(inventory["image_group"].value_counts(dropna=False))

# -------------------- 4. LOAD CASE DESCRIPTION TABLES --------------------

case_files = [
    "calc_case_description_train_set.csv",
    "calc_case_description_test_set.csv",
    "mass_case_description_train_set.csv",
    "mass_case_description_test_set.csv",
]

case_frames = []

for filename in case_files:
    path = CSV_DIR / filename
    if path.exists():
        frame = pd.read_csv(path)
        frame.columns = [str(c).strip() for c in frame.columns]
        frame["source_csv"] = filename
        case_frames.append(frame)
    else:
        print("Warning: case table not found:", filename)

if not case_frames:
    raise RuntimeError("No case-description CSV files were loaded.")

cases = pd.concat(case_frames, ignore_index=True)

patient_col = find_column(cases, ["patient_id"])
pathology_col = find_column(cases, ["pathology"])
full_col = find_column(
    cases, ["image file path", "image_file_path"]
)
crop_col = find_column(
    cases, ["cropped image file path", "cropped_image_file_path"]
)
roi_col = find_column(
    cases, ["ROI mask file path", "roi mask file path",
            "ROI_mask_file_path"]
)
laterality_col = find_column(
    cases, ["left or right breast", "left_or_right_breast"]
)
view_col = find_column(cases, ["image view", "image_view"])
abnormality_col = find_column(
    cases, ["abnormality type", "abnormality_type"]
)

if not all([patient_col, pathology_col, full_col]):
    raise ValueError(
        "Required case-table columns are missing. Available columns: "
        + str(cases.columns.tolist())
    )

print(f"\nCase-description records: {len(cases):,}")

# -------------------- 5. LABEL NORMALIZATION --------------------

def normalize_label(value):
    value = str(value).strip().upper()

    if value == "MALIGNANT":
        return "malignant"

    if value in {"BENIGN", "BENIGN_WITHOUT_CALLBACK"}:
        return "benign"

    return "unknown"

def extract_uids(source_path):
    if pd.isna(source_path):
        return []

    parts = str(source_path).replace("\\", "/").split("/")
    return [
        part for part in parts
        if part.startswith("1.3.6.1.4.1.")
    ]

# -------------------- 6. ROLE-AWARE IMAGE MAPPING --------------------

role_columns = {
    "full": full_col,
    "crop": crop_col,
    "roi": roi_col,
}

role_descriptions = {
    "full": "full mammogram images",
    "crop": "cropped images",
    "roi": "roi mask images",
}

def resolve_image(source_path, role):
    """
    Match a case CSV path to a JPEG using its series UID,
    the metadata role description, and filename agreement.

    Never choose an arbitrary candidate if the match is ambiguous.
    """
    uids = extract_uids(source_path)

    if not uids:
        return "", "no_uid_in_source_path"

    series_uid = uids[-1]
    candidates = uid_to_jpegs.get(series_uid, [])

    if not candidates:
        return "", "no_jpeg_for_series_uid"

    desired_description = role_descriptions[role]
    metadata_rows = metadata_by_uid.get(series_uid, [])

    role_rows = [
        r for r in metadata_rows
        if desired_description in r["description"]
    ]

    source_filename = os.path.basename(
        str(source_path).replace("\\", "/")
    )

    matched_paths = set()

    for metadata_row in role_rows:
        metadata_filename = metadata_row["filename"]

        # Prefer an exact filename match supported by metadata.
        if metadata_filename:
            for candidate in candidates:
                if os.path.basename(candidate) == metadata_filename:
                    matched_paths.add(candidate)

    if len(matched_paths) == 1:
        return next(iter(matched_paths)), "resolved"

    if len(matched_paths) > 1:
        return "", "ambiguous_role_candidates"

    return "", "role_not_identified_in_metadata"

# -------------------- 7. BUILD THE THREE-GROUP MANIFEST --------------------

records = []

for _, row in cases.iterrows():
    patient_id = (
        str(row[patient_col]).strip()
        if pd.notna(row[patient_col]) else ""
    )
    pathology = str(row[pathology_col]).strip()
    label = normalize_label(pathology)

    for role, path_column in role_columns.items():
        if path_column is None or pd.isna(row[path_column]):
            continue

        source_path = str(row[path_column]).strip()

        if not source_path:
            continue

        resolved_path, status = resolve_image(source_path, role)

        records.append({
            "role": role,
            "patient_id": patient_id,
            "pathology": pathology,
            "binary_label": label,
            "source_csv": row["source_csv"],
            "source_split": (
                "test" if "_test_set" in row["source_csv"]
                else "train"
            ),
            "laterality": (
                row[laterality_col]
                if laterality_col else ""
            ),
            "view": row[view_col] if view_col else "",
            "abnormality_type": (
                row[abnormality_col]
                if abnormality_col else ""
            ),
            "source_path": source_path,
            "resolved_path": resolved_path,
            "status": status,
        })

manifest = pd.DataFrame(records)

if manifest.empty:
    raise RuntimeError("No case-linked records were created.")

# -------------------- 8. VERIFY FILE EXISTENCE --------------------

manifest["file_exists"] = manifest["resolved_path"].apply(
    lambda p: bool(p) and Path(p).is_file()
)

# A resolved status is not sufficient if the file is missing.
manifest.loc[
    (manifest["status"] == "resolved") &
    (~manifest["file_exists"]),
    "status"
] = "resolved_file_missing"

# -------------------- 9. FIND LABEL CONFLICTS --------------------

valid = manifest[
    (manifest["status"] == "resolved") &
    (manifest["file_exists"])
].copy()

label_counts = (
    valid.groupby(["role", "resolved_path"])["binary_label"]
    .nunique()
)

conflict_keys = set(
    label_counts[label_counts > 1].index
)

manifest["label_conflict"] = [
    (row.role, row.resolved_path) in conflict_keys
    if row.resolved_path else False
    for row in manifest.itertuples()
]

conflict_report = manifest[
    manifest["label_conflict"]
].copy()

conflict_report.to_csv(
    OUT / "cbis_conflicting_label_records.csv", index=False
)

print("\nConflicting image paths by role:")
print(
    conflict_report.groupby("role")["resolved_path"]
    .nunique()
    .to_string()
    if not conflict_report.empty else "No conflicts detected."
)

# -------------------- 10. FLAG PATHS LINKED TO MULTIPLE PATIENTS --------------------

patient_counts = (
    valid.groupby(["role", "resolved_path"])["patient_id"]
    .nunique()
)

multi_patient_keys = set(
    patient_counts[patient_counts > 1].index
)

manifest["multiple_patient_ids"] = [
    (row.role, row.resolved_path) in multi_patient_keys
    if row.resolved_path else False
    for row in manifest.itertuples()
]

# -------------------- 11. BUILD CLEAN FULL-MAMMOGRAM MANIFEST --------------------

full = manifest[
    (manifest["role"] == "full") &
    (manifest["status"] == "resolved") &
    (manifest["file_exists"]) &
    (~manifest["label_conflict"]) &
    (~manifest["multiple_patient_ids"]) &
    (manifest["binary_label"].isin(["benign", "malignant"])) &
    (manifest["patient_id"].notna()) &
    (manifest["patient_id"].astype(str).str.strip() != "")
].copy()

# Exclude any path with a conflict anywhere in the resolved full-image records.
full_conflicts = (
    valid[valid["role"] == "full"]
    .groupby("resolved_path")["binary_label"]
    .nunique()
)
bad_full_paths = set(
    full_conflicts[full_conflicts > 1].index
)

full = full[~full["resolved_path"].isin(bad_full_paths)].copy()

# One image path = one classification example.
full = full.sort_values(
    ["resolved_path", "patient_id", "source_csv"]
).drop_duplicates(
    subset=["resolved_path"], keep="first"
).copy()

# Final checks
assert full["resolved_path"].is_unique
assert full["resolved_path"].map(
    lambda p: Path(p).is_file()
).all()
assert not full["label_conflict"].any()
assert not full["multiple_patient_ids"].any()

full.to_csv(
    OUT / "cbis_full_clean_unassigned.csv", index=False
)

# -------------------- 12. PREPARE CLEAN SUPPORTING GROUP MANIFESTS --------------------

# Crops and masks remain separate from the primary full-image classifier.
# Their case labels are retained for audit, but masks are not classifier inputs.
for role, output_name in [
    ("crop", "cbis_cropped_clean_unassigned.csv"),
    ("roi", "cbis_roi_clean_unassigned.csv"),
]:
    group = manifest[
        (manifest["role"] == role) &
        (manifest["status"] == "resolved") &
        (manifest["file_exists"]) &
        (~manifest["label_conflict"]) &
        (~manifest["multiple_patient_ids"])
    ].copy()

    # Keep only known labels for case-linked crops.
    # ROI masks are annotations; do not treat them as ordinary classifier inputs.
    if role == "crop":
        group = group[
            group["binary_label"].isin(["benign", "malignant"])
        ].copy()

    group = group.sort_values(
        ["resolved_path", "patient_id", "source_csv"]
    ).drop_duplicates("resolved_path")

    group.to_csv(OUT / output_name, index=False)

# Save complete resolved/unresolved manifest for audit.
manifest.to_csv(
    OUT / "cbis_all_groups_manifest_unassigned.csv", index=False
)

manifest[manifest["status"] != "resolved"].to_csv(
    OUT / "cbis_unresolved_records.csv", index=False
)

# -------------------- 13. FINAL AUDIT SUMMARY --------------------

summary_rows = []

for role in ["full", "crop", "roi"]:
    group = manifest[manifest["role"] == role]
    resolved = group[
        (group["status"] == "resolved") &
        (group["file_exists"])
    ]

    summary_rows.append({
        "group": role,
        "source_records": len(group),
        "resolved_records": len(resolved),
        "unique_resolved_paths": resolved["resolved_path"].nunique(),
        "unresolved_records": int((group["status"] != "resolved").sum()),
        "conflicting_paths": group.loc[
            group["label_conflict"], "resolved_path"
        ].nunique(),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "cbis_group_audit_summary.csv", index=False)

print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print("\nAll JPEG inventory:")
print(inventory["image_group"].value_counts(dropna=False))

print("\nCase-linked group summary:")
print(summary.to_string(index=False))

print("\nClean full mammogram dataset:")
print("Unique image paths:", full["resolved_path"].nunique())
print("Unique patients:", full["patient_id"].nunique())
print("\nImage-level labels:")
print(full["binary_label"].value_counts())

print("\nSource partition records (audit only; NOT final splits):")
print(full["source_split"].value_counts(dropna=False))

print("\nFiles with missing paths among retained full images:",
      sum(not Path(p).is_file() for p in full["resolved_path"]))

print("\nSaved outputs in:", OUT)
print("  cbis_all_jpeg_inventory.csv")
print("  cbis_all_groups_manifest_unassigned.csv")
print("  cbis_full_clean_unassigned.csv")
print("  cbis_cropped_clean_unassigned.csv")
print("  cbis_roi_clean_unassigned.csv")
print("  cbis_conflicting_label_records.csv")
print("  cbis_unresolved_records.csv")
print("  cbis_group_audit_summary.csv")

print("\nDONE: dataset audit and cleaning completed.")
print("Patient-level train/validation/test distribution has NOT been done.")
print("Next step: create leakage-free patient-level partitions.")

CBIS-DDSM DATASET PREPARATION

Total JPEG files discovered: 10,237

JPEG inventory by metadata group:
image_group
ambiguous       6286
full            2857
crop             744
unclassified     246
roi              104
Name: count, dtype: int64

Case-description records: 3,568

Conflicting image paths by role:
role
full    15

AUDIT SUMMARY

All JPEG inventory:
image_group
ambiguous       6286
full            2857
crop             744
unclassified     246
roi              104
Name: count, dtype: int64

Case-linked group summary:
group  source_records  resolved_records  unique_resolved_paths  unresolved_records  conflicting_paths
 full            3568              3286                   2857                 282                 15
 crop            3568              3566                   3566                   2                  0
  roi            3568              3246                   3246                 322                  0

Clean full mammogram dataset:
Unique image paths: 2842
U

In [10]:
# ================================================================
# CBIS-DDSM PATIENT-LEVEL TRAIN / VALIDATION / TEST DISTRIBUTION
# Uses the clean manifests created by the previous cell.
# 70% train / 15% validation / 15% test by patient.
# ================================================================

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

OUT = Path("/kaggle/working/cbis_ddsm_preparation")

# ---------- 1. Load the clean full-image manifest ----------
full_path = OUT / "cbis_full_clean_unassigned.csv"
all_path = OUT / "cbis_all_groups_manifest_unassigned.csv"
crop_path = OUT / "cbis_cropped_clean_unassigned.csv"
roi_path = OUT / "cbis_roi_clean_unassigned.csv"

for p in [full_path, all_path]:
    if not p.exists():
        raise FileNotFoundError(f"Required manifest missing: {p}")

full = pd.read_csv(full_path)
all_groups = pd.read_csv(all_path)

# Validate the full-image manifest.
full = full[
    full["binary_label"].isin(["benign", "malignant"])
].copy()

# Check missing values BEFORE converting to strings.
if full["patient_id"].isna().any() or full["resolved_path"].isna().any():
    raise ValueError("Missing patient IDs or image paths found.")

full["patient_id"] = full["patient_id"].astype(str).str.strip()
full["resolved_path"] = full["resolved_path"].astype(str)

if full["patient_id"].eq("").any() or full["patient_id"].str.lower().eq("nan").any():
    raise ValueError("Missing patient IDs found. Resolve before splitting.")
if not full["resolved_path"].map(lambda p: Path(p).is_file()).all():
    raise FileNotFoundError("Some clean full-image paths do not exist.")

if full["resolved_path"].duplicated().any():
    raise ValueError("Duplicate image paths found in the clean full manifest.")

if full["patient_id"].nunique() < 10:
    raise ValueError("Not enough patients to create reliable partitions.")

# ---------- 2. Build one stratification label per patient ----------
# This label is ONLY used to balance patient partitions.
# It does not change the image-level benign/malignant labels.
patient_table = (
    full.groupby("patient_id")["binary_label"]
    .apply(
        lambda labels: (
            "malignant" if (labels == "malignant").any()
            else "benign"
        )
    )
    .rename("patient_stratum")
    .reset_index()
)

print("Unique patients available:", len(patient_table))
print("Patient-level strata:")
print(patient_table["patient_stratum"].value_counts())

# ---------- 3. Split patients 70% / 15% / 15% ----------
train_patients, remaining_patients = train_test_split(
    patient_table,
    test_size=0.30,
    random_state=42,
    stratify=patient_table["patient_stratum"]
)

validation_patients, test_patients = train_test_split(
    remaining_patients,
    test_size=0.50,
    random_state=42,
    stratify=remaining_patients["patient_stratum"]
)

patient_to_split = {}

for patient_frame, split_name in [
    (train_patients, "train"),
    (validation_patients, "validation"),
    (test_patients, "test"),
]:
    for patient_id in patient_frame["patient_id"]:
        patient_to_split[patient_id] = split_name

# ---------- 4. Assign split to every clean full image ----------
full["new_split"] = full["patient_id"].map(patient_to_split)

if full["new_split"].isna().any():
    raise ValueError("Some full images did not receive a split.")

# ---------- 5. Apply the SAME patient assignment to all groups ----------
all_groups["patient_id"] = (
    all_groups["patient_id"].fillna("").astype(str).str.strip()
)
all_groups["new_split"] = all_groups["patient_id"].map(
    patient_to_split
)

# Records without a matching clean full-image patient remain unassigned.
# Do not randomly distribute them independently.
all_groups.to_csv(
    OUT / "cbis_all_groups_manifest_with_splits.csv",
    index=False
)

# Save final full-image manifest.
full.to_csv(
    OUT / "cbis_full_patient_split_manifest.csv",
    index=False
)

# Save one CSV per full-image split.
for split_name in ["train", "validation", "test"]:
    split_df = full[full["new_split"] == split_name].copy()
    split_df.to_csv(
        OUT / f"cbis_full_{split_name}.csv",
        index=False
    )

# Save crop and ROI records by patient split for separate experiments.
for role, source_path in [
    ("crop", crop_path),
    ("roi", roi_path),
]:
    if not source_path.exists():
        print(f"Note: {role} manifest not found; skipping.")
        continue

    group = pd.read_csv(source_path)
    group["patient_id"] = group["patient_id"].fillna("").astype(str).str.strip()
    group["new_split"] = group["patient_id"].map(patient_to_split)

    group.to_csv(
        OUT / f"cbis_{role}_with_patient_splits.csv",
        index=False
    )

    for split_name in ["train", "validation", "test"]:
        group[group["new_split"] == split_name].to_csv(
            OUT / f"cbis_{role}_{split_name}.csv",
            index=False
        )

# ---------- 6. Verify patient separation ----------
split_patients = {
    split_name: set(
        full.loc[full["new_split"] == split_name, "patient_id"]
    )
    for split_name in ["train", "validation", "test"]
}

overlap_tv = split_patients["train"] & split_patients["validation"]
overlap_tt = split_patients["train"] & split_patients["test"]
overlap_vt = split_patients["validation"] & split_patients["test"]

print("\n" + "=" * 65)
print("FINAL FULL-MAMMOGRAM DISTRIBUTION")
print("=" * 65)

print("\nImage counts by split and label:")
print(pd.crosstab(full["new_split"], full["binary_label"]))

print("\nUnique patients by split:")
print(full.groupby("new_split")["patient_id"].nunique())

print("\nImage totals by split:")
print(full["new_split"].value_counts())

print("\nPatient overlap checks:")
print("Train vs validation:", len(overlap_tv))
print("Train vs test:", len(overlap_tt))
print("Validation vs test:", len(overlap_vt))

# Check each patient is assigned to exactly one split.
assert full.groupby("patient_id")["new_split"].nunique().max() == 1
assert len(overlap_tv) == 0
assert len(overlap_tt) == 0
assert len(overlap_vt) == 0
assert full["resolved_path"].is_unique

print("\nPASS: no patient leakage between full-image partitions.")
print("\nOutput folder:", OUT)
print("Full manifest: cbis_full_patient_split_manifest.csv")
print("Split files: cbis_full_train.csv, cbis_full_validation.csv, cbis_full_test.csv")
print("Next step: image preprocessing and model data loaders.")

Unique patients available: 1457
Patient-level strata:
patient_stratum
benign       755
malignant    702
Name: count, dtype: int64

FINAL FULL-MAMMOGRAM DISTRIBUTION

Image counts by split and label:
binary_label  benign  malignant
new_split                      
test             247        196
train           1113        881
validation       220        185

Unique patients by split:
new_split
test           219
train         1019
validation     219
Name: patient_id, dtype: int64

Image totals by split:
new_split
train         1994
test           443
validation     405
Name: count, dtype: int64

Patient overlap checks:
Train vs validation: 0
Train vs test: 0
Validation vs test: 0

PASS: no patient leakage between full-image partitions.

Output folder: /kaggle/working/cbis_ddsm_preparation
Full manifest: cbis_full_patient_split_manifest.csv
Split files: cbis_full_train.csv, cbis_full_validation.csv, cbis_full_test.csv
Next step: image preprocessing and model data loaders.


In [18]:
# ============================================================
# GPU / CPU DIAGNOSTICS — RUN THIS CELL BEFORE TRAINING
# ============================================================
import time
import torch

print("=" * 60)
print("KAGGLE GPU DIAGNOSTICS")
print("=" * 60)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    torch.cuda.reset_peak_memory_stats()

    # Real GPU compute test
    x = torch.randn(1024, 1024, device=device)
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(10):
        x = x @ x
    torch.cuda.synchronize()
    print("GPU matrix computation: PASS")
    print("GPU test duration:", round(time.time() - start, 3), "seconds")
    print("GPU memory allocated:",
          round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")
    print("GPU memory reserved:",
          round(torch.cuda.memory_reserved() / 1024**3, 2), "GiB")
else:
    print("WARNING: CUDA is unavailable. In Kaggle, select a GPU accelerator.")

# Note: CPU is expected to handle image loading and OpenCV preprocessing.
# The model and image batches must be moved to CUDA during training.


KAGGLE GPU DIAGNOSTICS
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
GPU count: 2
GPU matrix computation: PASS
GPU test duration: 0.009 seconds
GPU memory allocated: 1.83 GiB
GPU memory reserved: 3.1 GiB


In [ ]:

import os
import time
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

CACHE_ROOT = "/kaggle/working/cbis_preprocessed_cache"
os.makedirs(CACHE_ROOT, exist_ok=True)

def build_cache(df, split_name):
    split_dir = os.path.join(CACHE_ROOT, split_name)
    os.makedirs(split_dir, exist_ok=True)

    records = []
    start = time.time()

    for idx, (_, row) in enumerate(df.iterrows()):
        cache_path = os.path.join(split_dir, f"{idx:05d}.png")

        if not os.path.exists(cache_path):
            img = preprocess_mammogram(row["resolved_path"])
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
            img.save(cache_path, optimize=False)

        records.append({
            "cached_path": cache_path,
            "binary_label": int(row["binary_label"])
        })

        if (idx + 1) % 100 == 0 or idx + 1 == len(df):
            print(
                f"{split_name}: {idx+1}/{len(df)} images cached",
                flush=True
            )

    print(
        f"{split_name} cache completed in "
        f"{(time.time()-start)/60:.1f} minutes"
    )
    return records


train_records = build_cache(train_df, "train")
val_records = build_cache(val_df, "validation")
test_records = build_cache(test_df, "test")


class CachedMammographyDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        with Image.open(record["cached_path"]) as im:
            image = im.convert("RGB")
        return self.transform(image), record["binary_label"]


train_ds = CachedMammographyDataset(
    train_records, train_transform
)
val_ds = CachedMammographyDataset(
    val_records, eval_transform
)
test_ds = CachedMammographyDataset(
    test_records, eval_transform
)

# Recreate loaders using cached images.
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Cached datasets and DataLoaders are ready.")


In [19]:
# ============================================================
# HYBRID MAMMOGRAPHY MODEL: Swin-T + ResNet50 + EfficientNet-B7
# CBIS-DDSM | Patient-level split | ROI preprocessing | XAI-ready
#
# Architecture combines your current Swin-T model with the paper's
# dual-CNN idea. This is NOT an exact reproduction of the paper:
# torchvision ResNet50 is used (not Keras ResNet50V2), and Swin-T is
# an additional branch. Results must be measured on your own splits.
#
# Expected CSV columns:
# resolved_path, binary_label, patient_id
# binary_label may be "benign"/"malignant" or 0/1.
# ============================================================

import os, time, copy, random, gc
import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    swin_t, Swin_T_Weights,
    resnet50, ResNet50_Weights,
    efficientnet_b7, EfficientNet_B7_Weights
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

# -------------------- CONFIG --------------------
PREP_DIR = "/kaggle/working/cbis_ddsm_preparation"
TRAIN_CSV = os.path.join(PREP_DIR, "cbis_full_train.csv")
VAL_CSV   = os.path.join(PREP_DIR, "cbis_full_validation.csv")
TEST_CSV  = os.path.join(PREP_DIR, "cbis_full_test.csv")
OUT_DIR = PREP_DIR
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = 256
BATCH_SIZE = 2            # Conservative for T4 with three backbones
EVAL_BATCH_SIZE = 2
HEAD_EPOCHS = 2
FINETUNE_EPOCHS = 12
PATIENCE = 4
NUM_WORKERS = 2
SEED = 42

# If the threshold ROI destroys too much image content, switch this off
# for an ablation comparison. Do not choose based on test-set results.
USE_THRESHOLD_ROI = True
THRESHOLD_VALUE = 193
USE_NLM_DENOISING = True
USE_MEDIAN_FILTER = True
USE_CLAHE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not os.path.isfile(p):
        raise FileNotFoundError(
            f"Missing manifest: {p}. Run your CBIS-DDSM preparation/split cell first."
        )

# -------------------- MANIFESTS + LABEL MAPPING --------------------
required = {"resolved_path", "binary_label", "patient_id"}

def prepare_manifest(df, name):
    df = df.copy()
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name} CSV missing columns: {missing}")

    df["resolved_path"] = df["resolved_path"].astype(str).str.strip()
    df["patient_id"] = df["patient_id"].astype(str).str.strip()

    label_map = {"benign": 0, "malignant": 1, "0": 0, "1": 1}
    raw = df["binary_label"].astype(str).str.strip().str.lower()
    mapped = raw.map(label_map)
    if mapped.isna().any():
        raise ValueError(
            f"{name} has unsupported labels: {raw[mapped.isna()].unique().tolist()}"
        )
    df["binary_label"] = mapped.astype(int)

    if df.empty:
        raise ValueError(f"{name} split is empty.")
    if df["binary_label"].nunique() != 2:
        raise ValueError(f"{name} must contain both benign and malignant examples.")
    if (df["resolved_path"] == "").any() or (df["patient_id"] == "").any():
        raise ValueError(f"{name} contains empty paths or patient IDs.")

    missing_files = [p for p in df["resolved_path"] if not os.path.isfile(p)]
    if missing_files:
        raise FileNotFoundError(
            f"{name}: {len(missing_files)} image files missing. Example: {missing_files[0]}"
        )

    print(
        f"{name:10s}: images={len(df)}, patients={df.patient_id.nunique()}, "
        f"benign={(df.binary_label==0).sum()}, malignant={(df.binary_label==1).sum()}"
    )
    return df.reset_index(drop=True)

train_df = prepare_manifest(pd.read_csv(TRAIN_CSV), "Train")
val_df   = prepare_manifest(pd.read_csv(VAL_CSV), "Validation")
test_df  = prepare_manifest(pd.read_csv(TEST_CSV), "Test")

tr_pat = set(train_df.patient_id.astype(str))
va_pat = set(val_df.patient_id.astype(str))
te_pat = set(test_df.patient_id.astype(str))
assert tr_pat.isdisjoint(va_pat), "Patient leakage: train/validation"
assert tr_pat.isdisjoint(te_pat), "Patient leakage: train/test"
assert va_pat.isdisjoint(te_pat), "Patient leakage: validation/test"
print("Patient-level split check: PASS")

# -------------------- PREPROCESSING --------------------
# Pipeline order: grayscale -> optional threshold ROI -> NLM -> median -> CLAHE
# This is an experimental adaptation of the described paper. A fixed threshold
# of 193 is intensity-scale dependent; inspect samples before trusting it.
def preprocess_mammogram(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        # Some dataset files may be readable by PIL even if OpenCV fails.
        with Image.open(path) as im:
            gray = np.asarray(im.convert("L"))

    if gray.dtype != np.uint8:
        gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    original = gray.copy()

    # Threshold ROI mask. Preserve original if threshold creates an unusably
    # tiny mask; do not silently turn a mammogram nearly black.
    if USE_THRESHOLD_ROI:
        _, mask = cv2.threshold(
            gray, THRESHOLD_VALUE, 255, cv2.THRESH_BINARY
        )
        foreground_fraction = float(np.count_nonzero(mask)) / mask.size
        if 0.005 <= foreground_fraction <= 0.95:
            gray = cv2.bitwise_and(gray, gray, mask=mask)
        else:
            gray = original

    if USE_NLM_DENOISING:
        gray = cv2.fastNlMeansDenoising(
            gray, None, h=5, templateWindowSize=7, searchWindowSize=21
        )

    if USE_MEDIAN_FILTER:
        gray = cv2.medianBlur(gray, 3)

    if USE_CLAHE:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)

    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(rgb)

# Mild augmentation only after patient-level splitting.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class MammographyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = preprocess_mammogram(row["resolved_path"])
            x = self.transform(img)
        except Exception as e:
            raise RuntimeError(f"Preprocessing failed: {row['resolved_path']}") from e
        return x, int(row["binary_label"])

train_ds = MammographyDataset(train_df, train_transform)
val_ds = MammographyDataset(val_df, eval_transform)
test_ds = MammographyDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda")
)

# -------------------- HYBRID MODEL --------------------
class HybridSwinCNN(nn.Module):
    """
    Swin-T pooled representation (768) +
    ResNet50 pooled representation (2048) +
    EfficientNet-B7 pooled representation (2560).
    Concatenated vector -> dropout 0.2 -> sigmoid-equivalent binary logits.
    """
    def __init__(self, pretrained=True):
        super().__init__()
        sw_weights = Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
        rn_weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        ef_weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None

        self.swin = swin_t(weights=sw_weights)
        self.swin.head = nn.Identity()  # output [B, 768]

        self.resnet = resnet50(weights=rn_weights)
        self.resnet.fc = nn.Identity()  # output [B, 2048]

        self.efficientnet = efficientnet_b7(weights=ef_weights)
        self.efficientnet.classifier = nn.Identity()  # output [B, 2560]

        self.dropout = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(768 + 2048 + 2560, 2)

    def forward(self, x):
        fs = self.swin(x)
        fr = self.resnet(x)
        fe = self.efficientnet(x)
        fused = torch.cat([fs, fr, fe], dim=1)
        return self.classifier(self.dropout(fused))

print("\nLoading pretrained Swin-T + ResNet50 + EfficientNet-B7...")
model = HybridSwinCNN(pretrained=True).to(device)
print("Model created.")
print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU allocated memory after model load:",
          round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")
print("Total parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=(device.type=="cuda"))

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_n = 0.0, 0
    ys, preds, probs_all = [], [], []

    for batch_idx, (xb, yb) in enumerate(loader):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Print device placement once on the first training batch.
        if is_train and batch_idx == 0:
            print(
                f"Batch device: {xb.device} | "
                f"Model device: {next(model.parameters()).device} | "
                f"CUDA allocated: "
                f"{torch.cuda.memory_allocated()/1024**3:.2f} GiB"
                if torch.cuda.is_available()
                else f"Batch device: {xb.device} | Model device: {next(model.parameters()).device}"
            )

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.autocast(device_type=device.type, enabled=(device.type=="cuda")):
                logits = model(xb)
                loss = criterion(logits, yb)
            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], max_norm=1.0
                )
                scaler.step(optimizer)
                scaler.update()

        n = yb.size(0)
        total_loss += loss.item() * n
        total_n += n
        prob = torch.softmax(logits.detach(), dim=1)[:, 1]
        pred = (prob >= 0.5).long()
        ys.extend(yb.detach().cpu().tolist())
        preds.extend(pred.cpu().tolist())
        probs_all.extend(prob.cpu().tolist())

    y = np.asarray(ys)
    p = np.asarray(preds)
    pr = np.asarray(probs_all)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0,1]).ravel()
    return {
        "loss": total_loss / max(total_n, 1),
        "accuracy": accuracy_score(y, p),
        "f1": f1_score(y, p, zero_division=0),
        "auc": roc_auc_score(y, pr),
        "precision": precision_score(y, p, zero_division=0),
        "sensitivity": tp/(tp+fn) if tp+fn else 0.0,
        "specificity": tn/(tn+fp) if tn+fp else 0.0
    }

# -------------------- PHASE 1: HEAD ONLY --------------------
for p in model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

head_optimizer = torch.optim.SGD(
    model.classifier.parameters(), lr=1e-4,
    momentum=0.9, nesterov=True, weight_decay=1e-4
)

history = []
print("\nPHASE 1: training fusion head")
for epoch in range(HEAD_EPOCHS):
    t0 = time.time()
    tr = run_epoch(model, train_loader, head_optimizer)
    va = run_epoch(model, val_loader)
    history.append({
        "phase":"head", "epoch":epoch+1,
        **{f"train_{k}":v for k,v in tr.items()},
        **{f"val_{k}":v for k,v in va.items()},
        "seconds":time.time()-t0
    })
    print(
        f"Head {epoch+1}/{HEAD_EPOCHS} | train loss {tr['loss']:.4f} "
        f"acc {tr['accuracy']:.3f} | val loss {va['loss']:.4f} "
        f"acc {va['accuracy']:.3f} AUC {va['auc']:.3f}"
    )

# -------------------- PHASE 2: FINE-TUNE ALL BACKBONES --------------------
# The paper description trains all layers. To reduce risk of catastrophic
# forgetting, use a small backbone LR and a larger head LR.
for p in model.parameters():
    p.requires_grad = True

backbone_params, head_params = [], []
for name, p in model.named_parameters():
    if name.startswith("classifier."):
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = torch.optim.SGD(
    [
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params, "lr": 1e-4}
    ],
    momentum=0.9,
    nesterov=True,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=FINETUNE_EPOCHS
)

best_val_accuracy = -float("inf")
best_val_auc_at_best_accuracy = float("nan")
best_state = None
best_epoch = None
bad_epochs = 0
checkpoint_path = os.path.join(OUT_DIR, "hybrid_swin_resnet50_effb7_best_accuracy.pth")

print("\nPHASE 2: fine-tuning hybrid backbones")
for epoch in range(FINETUNE_EPOCHS):
    t0 = time.time()
    tr = run_epoch(model, train_loader, optimizer)
    va = run_epoch(model, val_loader)
    scheduler.step()

    history.append({
        "phase":"finetune", "epoch":epoch+1,
        **{f"train_{k}":v for k,v in tr.items()},
        **{f"val_{k}":v for k,v in va.items()},
        "seconds":time.time()-t0
    })

    print(
        f"Epoch {epoch+1}/{FINETUNE_EPOCHS} | "
        f"train loss {tr['loss']:.4f} acc {tr['accuracy']:.3f} | "
        f"val loss {va['loss']:.4f} acc {va['accuracy']:.3f} "
        f"F1 {va['f1']:.3f} AUC {va['auc']:.3f} "
        f"Sens {va['sensitivity']:.3f} Spec {va['specificity']:.3f}"
    )

    # PRIMARY selection criterion: validation accuracy at threshold 0.50.
    # AUC is recorded for reporting only; it does not select the checkpoint.
    current_accuracy = va["accuracy"]
    improved = current_accuracy > best_val_accuracy

    # On equal accuracy, use validation loss as a tie-breaker.
    # This still prioritizes accuracy and avoids selecting by AUC.
    if (
        best_state is not None
        and np.isclose(current_accuracy, best_val_accuracy)
    ):
        previous_best_val_loss = best_val_loss
        improved = va["loss"] < previous_best_val_loss

    if improved:
        best_val_accuracy = current_accuracy
        best_val_loss = va["loss"]
        best_val_auc_at_best_accuracy = va["auc"]
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        torch.save({
            "model_state_dict": best_state,
            "architecture": "HybridSwinT_ResNet50_EfficientNetB7",
            "class_mapping": {"benign": 0, "malignant": 1},
            "image_size": IMG_SIZE,
            "threshold_roi": USE_THRESHOLD_ROI,
            "threshold_value": THRESHOLD_VALUE,
            "nlm": USE_NLM_DENOISING,
            "median": USE_MEDIAN_FILTER,
            "clahe": USE_CLAHE,
            "best_epoch": best_epoch,
            "best_validation_accuracy": best_val_accuracy,
            "validation_loss_at_best_accuracy": best_val_loss,
            "validation_auc_at_best_accuracy": best_val_auc_at_best_accuracy
        }, checkpoint_path)
        bad_epochs = 0
        print("  Saved best validation-accuracy checkpoint.")
    else:
        bad_epochs += 1
        print(f"  No validation-accuracy improvement ({bad_epochs}/{PATIENCE}).")
        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

pd.DataFrame(history).to_csv(
    os.path.join(OUT_DIR, "hybrid_swin_resnet50_effb7_history.csv"),
    index=False
)

if best_state is None:
    raise RuntimeError("No checkpoint selected; inspect training/validation output.")

model.load_state_dict(best_state)
print("\nBest validation accuracy:", round(best_val_accuracy, 4))
print("Validation AUC at selected checkpoint (reporting only):",
      round(float(best_val_auc_at_best_accuracy), 4))
print("Best epoch:", best_epoch)
print("Checkpoint:", checkpoint_path)

# -------------------- FINAL TEST: RUN ONCE AFTER MODEL SELECTION --------------------
print("\nFINAL HELD-OUT TEST EVALUATION (threshold=0.50)")
test_metrics = run_epoch(model, test_loader)
for k, v in test_metrics.items():
    print(f"{k:15s}: {v:.4f}")

ys, preds, probs_all = [], [], []
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=(device.type=="cuda")):
            logits = model(xb)
        prob = torch.softmax(logits, dim=1)[:, 1]
        pred = (prob >= 0.5).long()
        ys.extend(yb.numpy().tolist())
        preds.extend(pred.cpu().tolist())
        probs_all.extend(prob.cpu().tolist())

print("\nClassification report (0=benign, 1=malignant)")
print(classification_report(
    ys, preds, labels=[0,1],
    target_names=["Benign", "Malignant"], zero_division=0
))
print("Confusion matrix [true rows, predicted columns]:")
print(confusion_matrix(ys, preds, labels=[0,1]))

print("\nSaved files:")
print(checkpoint_path)
print(os.path.join(OUT_DIR, "hybrid_swin_resnet50_effb7_history.csv"))
print("\nImportant: 98.13% in the paper is not guaranteed to reproduce here.")
print("The threshold ROI is an experiment; compare against a no-threshold run")
print("using validation data only. Keep the test set untouched during tuning.")


Device: cuda
GPU: Tesla T4
Train     : images=1994, patients=1019, benign=1113, malignant=881
Validation: images=405, patients=219, benign=220, malignant=185
Test      : images=443, patients=219, benign=247, malignant=196
Patient-level split check: PASS

Loading pretrained Swin-T + ResNet50 + EfficientNet-B7...
Model created.
Model device: cuda:0
CUDA available: True
GPU: Tesla T4
GPU allocated memory after model load: 2.27 GiB
Total parameters: 114,825,100

PHASE 1: training fusion head
Batch device: cuda:0 | Model device: cuda:0 | CUDA allocated: 2.28 GiB


KeyboardInterrupt: 

In [21]:

import time
import torch
from torch.utils.data import DataLoader

print("1. Testing one image preprocessing...", flush=True)
t0 = time.time()

img, label = train_ds[0]

print(
    f"PASS: image loaded in {time.time()-t0:.1f}s | "
    f"shape={tuple(img.shape)} | label={label}",
    flush=True
)

print("\n2. Testing DataLoader with workers disabled...", flush=True)
debug_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

t0 = time.time()
xb, yb = next(iter(debug_loader))
print(f"PASS: batch loaded in {time.time()-t0:.1f}s", flush=True)

print("\n3. Testing hybrid model forward pass on GPU...", flush=True)
model.eval()
xb = xb.to("cuda", non_blocking=True)

torch.cuda.synchronize()
t0 = time.time()

with torch.no_grad(), torch.autocast(
    device_type="cuda", dtype=torch.float16
):
    output = model(xb)

torch.cuda.synchronize()

print("PASS: output shape =", tuple(output.shape), flush=True)
print("Forward pass time:", round(time.time()-t0, 2), "seconds")
print("GPU allocated memory:",
      round(torch.cuda.memory_allocated()/1024**3, 2), "GiB")
print("\nAll diagnostic stages completed.")


1. Testing one image preprocessing...
PASS: image loaded in 12.6s | shape=(3, 256, 256) | label=0

2. Testing DataLoader with workers disabled...
PASS: batch loaded in 12.8s

3. Testing hybrid model forward pass on GPU...
PASS: output shape = (1, 2)
Forward pass time: 5.08 seconds
GPU allocated memory: 2.28 GiB

All diagnostic stages completed.


In [11]:
import time, random, warnings
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", category=UserWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---------------- CONFIG ----------------
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 4
MAX_EPOCHS = 10
PATIENCE = 4
NUM_WORKERS = 2
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.0
LR_BODY = 2e-5
LR_PATCH_PROJ = 2e-5
LR_HEAD = 1e-4

OUTPUT_DIR = Path("/kaggle/working/cbis_ddsm_preparation")
TRAIN_CSV = OUTPUT_DIR / "cbis_full_train.csv"
VAL_CSV = OUTPUT_DIR / "cbis_full_validation.csv"
TEST_CSV = OUTPUT_DIR / "cbis_full_test.csv"
CHECKPOINT_PATH = OUTPUT_DIR / "vit_b16_finetuned_best.pth"
HISTORY_PATH = OUTPUT_DIR / "vit_b16_finetuned_history.csv"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = device.type == "cuda"
PIN_MEMORY = device.type == "cuda"
print("Device:", device)

# ---------------- LOAD SPLITS ----------------
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

LABEL_MAP = {"benign": 0, "malignant": 1, "0": 0, "1": 1}
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    enc = df["binary_label"].astype(str).str.strip().str.lower().map(LABEL_MAP)
    if enc.isna().any():
        raise ValueError(f"{name}: unknown labels")
    df["binary_label"] = enc.astype(int)
    df["resolved_path"] = df["resolved_path"].astype(str)
    df["patient_id"] = df["patient_id"].astype(str)
    missing = [p for p in df["resolved_path"] if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(f"{name}: {len(missing)} missing files, e.g. {missing[0]}")

tp_, vp_, sp_ = (set(d["patient_id"]) for d in (train_df, val_df, test_df))
assert not (tp_ & vp_) and not (tp_ & sp_) and not (vp_ & sp_), "Patient leakage"
assert train_df["binary_label"].nunique() == 2 and val_df["binary_label"].nunique() == 2

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print(train_df["binary_label"].map({0: "benign", 1: "malignant"}).value_counts().to_string())

# ---------------- DATA ----------------
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class MammographyDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row["resolved_path"]) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")
            img = self.transform(img)
        return img, torch.tensor(int(row["binary_label"]), dtype=torch.long)

def make_loader(df, tfm, bs, shuffle):
    return DataLoader(MammographyDataset(df, tfm), batch_size=bs, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      persistent_workers=(NUM_WORKERS > 0))

train_loader = make_loader(train_df, train_transform, BATCH_SIZE, True)
val_loader = make_loader(val_df, eval_transform, EVAL_BATCH_SIZE, False)

# ---------------- MODEL (full fine-tuning) ----------------
model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
model.heads.head = nn.Linear(model.heads.head.in_features, 2)
model = model.to(device)

for p in model.parameters():
    p.requires_grad = True
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable: {sum(p.numel() for p in trainable_parameters):,}")

# ---------------- OPTIMIZER (covers EVERY parameter) ----------------
head_params = list(model.heads.parameters())
proj_params = list(model.conv_proj.parameters())
special_ids = {id(p) for p in head_params + proj_params}
body_params = [p for p in model.parameters() if id(p) not in special_ids]  # blocks, ln, class_token, pos_embedding
assert len(head_params) + len(proj_params) + len(body_params) == len(list(model.parameters()))

optimizer = torch.optim.AdamW(
    [{"params": body_params, "lr": LR_BODY},
     {"params": proj_params, "lr": LR_PATCH_PROJ},
     {"params": head_params, "lr": LR_HEAD}],
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = len(train_loader)
total_steps = MAX_EPOCHS * steps_per_epoch
warmup_steps = steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

cw = compute_class_weight("balanced", classes=np.array([0, 1]),
                          y=train_df["binary_label"].to_numpy())
class_weights = torch.tensor(cw, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
print("Class weights [benign, malignant]:", cw.tolist())

def fmt(s):
    return str(timedelta(seconds=max(0, int(s))))

# ---------------- TRAIN / VALIDATE ----------------
def train_one_epoch():
    model.train()
    total, n = 0.0, 0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ENABLED):
            loss = criterion(model(images), labels)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss.item()}")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item() * labels.size(0)
        n += labels.size(0)
    return total / max(n, 1)

@torch.no_grad()
def validate_model():
    model.eval()
    total, n = 0.0, 0
    ys, ps = [], []
    for images, labels in val_loader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(images)
            loss = criterion(logits, labels)
        total += loss.item() * labels.size(0)
        n += labels.size(0)
        ps.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
        ys.append(labels.cpu().numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "val_loss": float(total / max(n, 1)),
        "val_accuracy": float(accuracy_score(y, pred)),
        "val_f1": float(f1_score(y, pred, zero_division=0)),
        "val_auc": float(roc_auc_score(y, p)),
        "val_precision": float(precision_score(y, pred, zero_division=0)),
        "val_recall": float(recall_score(y, pred, zero_division=0)),
        "val_specificity": float(tn / max(tn + fp, 1)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ---------------- MAIN LOOP ----------------
history, best_auc, no_improve = [], -1.0, 0
t0 = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    te = time.perf_counter()
    train_loss = train_one_epoch()
    m = validate_model()
    history.append({"epoch": epoch, "train_loss": train_loss, **m,
                    "lr_body": optimizer.param_groups[0]["lr"],
                    "lr_head": optimizer.param_groups[2]["lr"]})
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | train {train_loss:.4f} | val loss {m['val_loss']:.4f} | "
          f"acc {m['val_accuracy']:.4f} | F1 {m['val_f1']:.4f} | AUC {m['val_auc']:.4f} | "
          f"sens {m['val_recall']:.4f} | spec {m['val_specificity']:.4f} | "
          f"epoch {fmt(time.perf_counter() - te)} | elapsed {fmt(time.perf_counter() - t0)}")

    if m["val_auc"] > best_auc + 1e-4:
        best_auc, no_improve = m["val_auc"], 0
        torch.save({"epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "best_val_auc": best_auc,
                    "image_size": IMAGE_SIZE,
                    "class_to_idx": {"benign": 0, "malignant": 1}},
                   CHECKPOINT_PATH)
        print(f"  saved best (val AUC {best_auc:.4f})")
    else:
        no_improve += 1
        print(f"  no improvement {no_improve}/{PATIENCE}")
        if no_improve >= PATIENCE:
            print("Early stopping.")
            break

ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"\nRestored best epoch {ckpt['epoch']} (val AUC {ckpt['best_val_auc']:.4f}). Test set not yet used.")

Device: cuda
Train/Val/Test: 1994 405 443
binary_label
benign       1113
malignant     881
Trainable: 85,800,194
Class weights [benign, malignant]: [0.8957771787960467, 1.1316685584562998]
Epoch 01/10 | train 0.6641 | val loss 0.6093 | acc 0.6420 | F1 0.5246 | AUC 0.7523 | sens 0.4324 | spec 0.8182 | epoch 0:04:02 | elapsed 0:04:02
  saved best (val AUC 0.7523)


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import roc_curve

test_loader = DataLoader(MammographyDataset(test_df, eval_transform),
                         batch_size=EVAL_BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

@torch.no_grad()
def collect_probs(loader):
    model.eval()
    ys, ps = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(images)
        ps.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
        ys.append(labels.numpy())
    return np.concatenate(ys), np.concatenate(ps)

# Threshold chosen on validation only (Youden's J)
yv, pv = collect_probs(val_loader)
fpr, tpr, thr = roc_curve(yv, pv)
best_thr = float(thr[np.argmax(tpr - fpr)])

yt, pt = collect_probs(test_loader)

def report(y, p, t, name):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    print(f"\n{name} (threshold={t:.3f})")
    print(f"Accuracy   : {accuracy_score(y, pred):.4f}")
    print(f"Precision  : {precision_score(y, pred, zero_division=0):.4f}")
    print(f"Recall/Sens: {recall_score(y, pred, zero_division=0):.4f}")
    print(f"Specificity: {tn / max(tn + fp, 1):.4f}")
    print(f"F1         : {f1_score(y, pred, zero_division=0):.4f}")
    print(f"AUC        : {roc_auc_score(y, p):.4f}")
    print(f"Confusion [TN, FP, FN, TP]: [{tn}, {fp}, {fn}, {tp}]")

report(yt, pt, 0.5, "TEST @ 0.5")
report(yt, pt, best_thr, "TEST @ validation-tuned threshold")

# Patient-level bootstrap CI for AUC
rng = np.random.default_rng(SEED)
pids = test_df["patient_id"].to_numpy()
groups = {p: np.where(pids == p)[0] for p in np.unique(pids)}
keys = list(groups)
aucs = []
for _ in range(1000):
    sampled = rng.choice(keys, size=len(keys), replace=True)
    idx = np.concatenate([groups[k] for k in sampled])
    if len(np.unique(yt[idx])) == 2:
        aucs.append(roc_auc_score(yt[idx], pt[idx]))
print(f"\nTest AUC 95% CI (patient bootstrap): [{np.percentile(aucs, 2.5):.4f}, {np.percentile(aucs, 97.5):.4f}]")